# 06 — Equal risk contribution

`EqualRiskContribution`, the long-only equal-risk-contribution portfolio, against EW and 60/40; hypotheses are pre-registered in §2 and cross-model tests are in notebooks 08–09.

## §1 Risk parity and equal risk contribution

Risk parity allocates by risk instead of capital. In the equal-risk-contribution (ERC) form every asset carries the same share of portfolio variance (Maillard, Roncalli & Teïletche 2010). ERC is the minimum-variance portfolio subject to a floor on the sum of log weights. Qian (2006) reads risk contributions as contributions to large losses. Spinu (2013) writes long-only ERC as a strictly convex problem with a unique solution, and Griveau-Billion, Richard & Roncalli (2013) solve it by cyclical coordinate descent, as the class does. Asness, Frazzini & Pedersen (2012) study levered risk parity; the ERC here is fully invested and unlevered.

Notebooks 04–05 found a UUP corner in GMV and MDP driven by correlation. ERC caps every asset's variance share at 1/N. The question is what that cap does to UUP's capital weight.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
import sys
from pathlib import Path
import inspect
import textwrap
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb06_helpers as h
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, MostDiversified,
    EqualRiskContribution, FixedWeight, risk_contributions, backtest, summary, mdp_closed_form,
)
from maplab.models import _min_variance_long_only, _mdp_long_only, _erc_long_only, _tangency_long_only

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

In [ ]:
ml.inference.render_registration(ml.inference.load_registration("nb06"))

## §3 The math

**Risk contributions.** Portfolio volatility $\sigma_p = \sqrt{w'\Sigma w}$ is homogeneous of degree 1 in $w$, so Euler's theorem splits it exactly: $\sigma_p = \sum_i RC_i$ with $RC_i = w_i(\Sigma w)_i/\sigma_p$. Asset $i$'s variance share is $RC_i/\sigma_p = w_i\beta_{i,p}$, where $\beta_{i,p} = (\Sigma w)_i/(w'\Sigma w)$ is its beta to the portfolio.

**Risk budgeting (general form).** Given budgets $b_i > 0$ with $\sum_i b_i = 1$, find $w \ge 0$, $\mathbf 1'w = 1$ with $w_i\beta_{i,p} = b_i$ for every $i$. The long-only solution is unique and solves

$$\min_{y>0}\ \tfrac12\, y'\Sigma y - \sum_i b_i \log y_i, \qquad w = y/\mathbf 1'y .$$

The first-order condition $y_i(\Sigma y)_i = b_i$ delivers the budgets exactly. The log term keeps every weight strictly positive, so there is no boundary case and no fallback.

**ERC** sets $b_i = 1/N$; the class has equal budgets only. Two identities follow.

- Every variance share is $1/N$: $w_i\beta_{i,p} = 1/N$. With $\beta_{i,p} = \rho_{i,p}\sigma_i/\sigma_p$, $\;w_i = \dfrac{\sigma_p}{N\,\sigma_i\,\rho_{i,p}}$, so an asset gets a large capital weight if its volatility is low, its correlation with the portfolio is low, or both. Every $\beta_{i,p}$ is positive: no asset hedges an ERC portfolio.
- The risk weights $x = w\circ\sigma/(w'\sigma)$ are ERC on the correlation matrix $C = D^{-1}\Sigma D^{-1}$, so ERC is scale-invariant, like MDP in notebook 05.

**Special cases.** With equal correlations (any $N$), or with $N = 2$, ERC is inverse-vol: $w_i \propto 1/\sigma_i$.

**Contrast with GMV.** Long-only GMV's first-order condition gives $\beta_{i,p} = 1$ on every held asset: GMV equalizes marginal risk over the assets it holds, ERC equalizes variance shares over all 13. Volatility is ordered $\sigma_{GMV} \le \sigma_{ERC} \le \sigma_{EW}$.

## §4 The class and the rolling backtest

`EqualRiskContribution` estimates Σ with `_estimate` (notebook 01 §3) and passes it to `_erc_long_only`. Each coordinate-descent sweep solves $\Sigma_{ii}y_i^2 + c_i y_i - 1/N = 0$ for its positive root, with $c_i = \sum_{j\ne i}\Sigma_{ij}y_j$, starting from inverse vol. It stops when every $y_i(\Sigma y)_i$ is within $10^{-12}$ of $1/N$ and raises after 1,000 sweeps.

In [ ]:
for obj in (EqualRiskContribution.predict_weights, _erc_long_only):
    print(f"# {obj.__module__}.{obj.__qualname__}")
    print(textwrap.dedent(inspect.getsource(obj)))

### Backtests

Seven runs, harness as in notebook 01 §3. ERC(S) is registered, ERC(LW) is the sensitivity and IV the reference (§2's notebook-local IV is now the package class `InverseVol`, same weights); GMV(S), EW, MDP(S) and 60/40 are the reproduction gate, and EW and 60/40 are the benchmarks.

In [ ]:
strategies = {
    "ERC(S)": EqualRiskContribution(cov_estimator=ml.sample_cov),
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "ERC(LW)": EqualRiskContribution(cov_estimator=ml.ledoit_wolf_cov),
    "IV": ml.InverseVol(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

GMV(S), EW, MDP(S) and 60/40 must match notebooks 01 and 05 to 2 dp in every window; otherwise the cell stops the notebook.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4

In [ ]:
ml.inference.check_reproduction(summary_table, {"GMV(S)": "GMV_sample", "EW": "EW", "MDP(S)": "MDP_sample",
                                                "60/40": "SixtyForty_SPY_IEF"})

print("REPRODUCTION GATE: PASSED -- GMV(S), EW, MDP(S), 60/40 match notebooks 01/05 to 2 dp.")

### Solver checks

At all 208 rebalances, under the sample and LW Σ, recomputed with `_erc_long_only`: variance shares equal to 1/13, the Euler sum, both §3 identities, the equal-correlation case, $\sigma_{GMV} \le \sigma_{ERC} \le \sigma_{EW}$ and positive weights, to 1e-11 (the solver stops at 1e-12).

In [ ]:
mech5_table = h.mechanism_checks(panel, rdates)
print(mech5_table.to_string(float_format=lambda v: f"{v:.3e}"))

for _, row in mech5_table.iterrows():
    assert row["max_rc_share_dev"] <= 1e-11, f"RC-share deviation failed: {row['max_rc_share_dev']}"
    assert row["max_euler"] <= 1e-12, f"Euler check failed: {row['max_euler']}"
    assert row["max_identA"] <= 1e-11, f"Identity A failed: {row['max_identA']}"
    assert row["max_identB"] <= 1e-11, f"Identity B failed: {row['max_identB']}"
    assert row["max_rho_bar_dev"] <= 1e-11, f"constant-correlation check failed: {row['max_rho_bar_dev']}"
    assert row["min_vol_margin_gmv"] >= -1e-11, f"vol ordering (GMV<=ERC) failed: {row['min_vol_margin_gmv']}"
    assert row["min_vol_margin_ew"] >= -1e-11, f"vol ordering (ERC<=EW) failed: {row['min_vol_margin_ew']}"
    assert row["min_w"] > 0, f"positivity failed: {row['min_w']}"

print("Mechanism checks: PASSED at all 208 rebalances, both estimators.")

n_sweeps_s = results["ERC(S)"]["strat"].sweeps
n_sweeps_lw = results["ERC(LW)"]["strat"].sweeps
print(f"strat.sweeps: ERC(S) median={np.median(n_sweeps_s):.0f} max={max(n_sweeps_s)}   "
      f"ERC(LW) median={np.median(n_sweeps_lw):.0f} max={max(n_sweeps_lw)}")

All checks pass for both estimators; the largest variance-share deviation is 1.2e-12. Coordinate descent needs a median of 20 sweeps (max 53) on the sample Σ and 19 (max 48) on LW.

## §5 Snapshots

### 5.1 Three dates

ERC, GMV, MDP, MaxSharpe, EW and IV on the sample Σ at the three dates that fixed the H1/H2 directions. MaxSharpe appears in snapshots only; it is not backtested.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

snap_table, uup_context_table, gmv_beta_dev = h.snapshot_tables(panel, DATES)
print(snap_table.round(4).to_string())

print("\nUUP context (vol rank 1 = lowest; rho_bar = mean off-diagonal correlation of all 13):")
print(uup_context_table.round(4))

print("\nGMV first-order check: max|beta_i,p - 1| over held assets (w_i > 1e-8):")
for d, v in gmv_beta_dev.items():
    print(f"  {d}: {v:.3e}")

At all three dates ERC holds less UUP than GMV and MDP and has a higher effective N than both, the directions registered for H1 and H2. Under GMV every held asset has $\beta_{i,p} = 1$ (to 5.2e-05, SLSQP precision), so UUP's variance share equals its capital weight: 0.5812 at 2022-12-31. Under ERC it is 1/13.

### 5.2 Snapshot map at 2022-12-31

ERC, GMV, MDP, MaxSharpe, EW and 60/40 in (volatility, μ) space with the long-only frontier. μ is the 252-day sample mean, so read the x-axis.

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()
n = Sigma_np.shape[0]
tickers = list(Sigma_2022.columns)

w_gmv_2022 = _min_variance_long_only(Sigma_2022)
w_mdp_2022, _ = _mdp_long_only(Sigma_2022, "F1", asof_2022)
w_erc_2022, _ = _erc_long_only(Sigma_2022, "F1", asof_2022)
rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "F1", asof_2022)
w_ew_2022 = np.full(n, 1.0 / n)
w_6040_2022 = FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, asof_2022).reindex(tickers).to_numpy()

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022,
    {"ERC": w_erc_2022, "GMV": w_gmv_2022, "MDP": w_mdp_2022,
     "MaxSharpe": w_msr_2022, "EqualWeight": w_ew_2022, "60/40": w_6040_2022},
    title=f"Snapshot {asof_2022.date()} -- ERC / GMV / MDP / MaxSharpe / EW / 60/40",
    figsize=(10, 6), legend_outside=True,
    styles={"60/40": dict(marker="D", s=130, color=ml.plotting.METHOD_COLORS["60/40"])},
)
plt.show()

iU2022 = tickers.index("UUP")
print(f"w_UUP: GMV={w_gmv_2022[iU2022]:.4f}  MDP={w_mdp_2022[iU2022]:.4f}  ERC={w_erc_2022[iU2022]:.4f}  "
      f"MaxSharpe={w_msr_2022[iU2022]:.4f}  EW={w_ew_2022[iU2022]:.4f}")

ERC sits just right of GMV and MDP at the low-volatility end (ex-ante 4.32% against 3.32% and 3.50%), well left of EW (11.04%). Its UUP weight, 48.36%, is below GMV's 58.12%, MDP's 59.22% and MaxSharpe's 74.41%, and far above EW's 7.69%.

### 5.3 Capital and variance share at the three dates

Capital weight (dark) and variance share (light) per asset for GMV, MDP, ERC and EW; UUP in red. ERC's variance shares sit on the 1/13 line by construction; the gap between UUP's two bars is the corner.

In [ ]:
fig, grid_table = h.mechanism_grid(panel, DATES)
plt.show()

GMV's two bars match on every held asset ($\beta_{i,p} = 1$); MDP's do not. ERC's variance shares sit on 1/13 while UUP's capital bar stands at 25.74%, 24.14% and 48.36%. Under EW, UUP's variance share is negative at all three dates (−0.0154, −0.0364, −0.0419, §5.1 table): it hedges EW.

### 5.4 The ERC identity at 2022-12-31

Variance shares, betas to the portfolio and the weight formula $w_i = \sigma_p/(N\sigma_i\rho_{i,p})$ at the solved ERC weights.

In [ ]:
S_22 = Sigma_2022.to_numpy()
sig_22 = np.sqrt(np.diag(S_22))
var_22 = float(w_erc_2022 @ S_22 @ w_erc_2022)
sp_22 = np.sqrt(var_22)
beta_22 = (S_22 @ w_erc_2022) / var_22
rho_22 = (S_22 @ w_erc_2022) / (sig_22 * sp_22)
share_22 = w_erc_2022 * beta_22
w_formula_22 = sp_22 / (n * sig_22 * rho_22)
others = [i for i in range(n) if i != iU2022]
kkt = dict(
    vs_dev=float(np.max(np.abs(share_22 - 1.0 / n))),
    wf_dev=float(np.max(np.abs(w_formula_22 - w_erc_2022))),
    beta_min=float(beta_22.min()), beta_min_name=tickers[int(np.argmin(beta_22))],
    beta_uup=float(beta_22[iU2022]), rho_uup=float(rho_22[iU2022]),
    w_uup=float(w_erc_2022[iU2022]), nw_uup=float(n * w_erc_2022[iU2022]),
    vol_f=float(sp_22 / sig_22[iU2022]), vol_f_med=float(np.median(sp_22 / sig_22[others])),
    corr_f=float(1.0 / rho_22[iU2022]), corr_f_med=float(np.median(1.0 / rho_22[others])),
)
print(f"max |variance share - 1/N| = {kkt['vs_dev']:.3e}")
print(f"max |w - sigma_p/(N sigma_i rho_i,p)| = {kkt['wf_dev']:.3e}")
print(f"min beta_i,p = {kkt['beta_min']:.6f} ({kkt['beta_min_name']});  UUP beta_i,p = {kkt['beta_uup']:.6f}")
print(f"UUP: w = {kkt['w_uup']:.6f}  N*w = {kkt['nw_uup']:.6f}  sigma_p/sigma_i = {kkt['vol_f']:.6f}  "
      f"1/rho_i,p = {kkt['corr_f']:.6f}  rho_i,p = {kkt['rho_uup']:.6f}")
print(f"other 12, median: sigma_p/sigma_i = {kkt['vol_f_med']:.6f}  1/rho_i,p = {kkt['corr_f_med']:.6f}")

Every variance share equals 1/13 to 4.6e-13 and the weights match $\sigma_p/(N\sigma_i\rho_{i,p})$ to 1.1e-12; all 13 betas to the portfolio are positive, the smallest UUP's 0.1591. UUP's weight, 48.36%, is 6.29 × 1/13 = 0.466 ($\sigma_p/\sigma_i$; median of the other 12: 0.246) × 13.49 ($1/\rho_{i,p}$, with $\rho_{i,p}$ = 0.0741; median 1.91). Low volatility and low correlation with the portfolio both raise the weight, correlation by more.

## §6 Out-of-sample results

### 6.1 Summary

Walk-forward as in notebook 01 §5; the test window (2023+) is the cleaner read, and wealth paths and formal tests are in notebook 08.

In [ ]:
cols = ["ann_return", "ann_vol", "sharpe", "max_dd", "ann_turnover"]
group_order = ["registered", "sensitivity", "reference", "reproduction gate", "benchmark"]
window_order = ["full", "train (<= split)", "test (> split)"]
summary_table2 = summary_table[cols].reset_index()
summary_table2["group"] = pd.Categorical(summary_table2["strategy"].map(h.run_group), group_order, ordered=True)
assert summary_table2["group"].notna().all()
summary_table2["strategy"] = pd.Categorical(summary_table2["strategy"], list(results), ordered=True)
summary_table2["window"] = pd.Categorical(summary_table2["window"], window_order, ordered=True)
summary_table2 = summary_table2.set_index(["group", "strategy", "window"]).sort_index()
print(summary_table2.round(4).to_string())

ERC(S) runs at 3.94% volatility, between GMV(S)'s 3.17% and EW's 7.87% as §3 orders them, with turnover of 27%/yr (EW 14%, MDP(S) 52%). Its full-window Sharpe ratio is 0.76 (GMV(S) 0.62, EW 0.74, 60/40 0.91), and 0.94 in the test window against 60/40's 1.00.

### 6.2 Paired differences (H3–H5)

For the five registered pairs: daily return differences, each run's CAPM α and β, the regression of each difference on the market (Δα with its own t), and the β split total = β part + Δα; plain OLS, no HAC (notebook 08 uses Newey–West).

In [ ]:
pairs = [
    ("ERC(S)", "GMV(S)"),
    ("ERC(S)", "MDP(S)"),
    ("ERC(S)", "EW"),
    ("ERC(LW)", "ERC(S)"),
    ("ERC(S)", "IV"),
]

paired_table, paired_series = ml.inference.paired_table(results, pairs, split_ts, summary_table)
print(paired_table[["ann_mean_diff", "TE", "t", "cost_drag_diff"]].round(4).to_string())

ERC(S) − GMV(S) is +1.04%/yr (t 2.47) full but t 1.32 test; ERC(S) − MDP(S) is +0.40%/yr (t 0.99). ERC(S) − EW is −2.78%/yr (t −2.08) and −3.16%/yr in the test window; ERC(LW) − ERC(S) has t 1.75 full and 1.24 test. Cost-drag differences are at most 0.05%/yr, so costs change no sign.

Single-strategy CAPM α and β, 60/40 included, are context only and not registered; notebook 08 tests all 23 α jointly.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, ["ERC(S)", "GMV(S)", "MDP(S)", "EW", "60/40", "ERC(LW)", "IV"],
                                     MKT_simple, rf_daily)
print(capm_table.round(4))

In [ ]:
# Δα with its own t (the H3/H4/H5 statistic), not α_a − α_b
diff_capm_table = ml.inference.diff_regression(paired_series, pairs, MKT_simple)
print(diff_capm_table.round(4))

In [ ]:
decomp = ml.inference.beta_decomposition(capm_table, paired_table, diff_capm_table, pairs,
                                         results, rf_daily, MKT_simple)
print(ml.inference.format_beta_decomposition(decomp, capm_table, diff_capm_table), end="")

β split (full window): ERC(S) − GMV(S) is β part +0.95%/yr plus Δα +0.09%/yr (t 0.29), so the gap is ERC's higher β, which H3 sets aside by testing Δα. ERC(S) − EW's β part, −3.53%/yr, exceeds the whole −2.78%/yr shortfall, against Δα +0.76%/yr (t 0.89). ERC(S) − MDP(S) and ERC(S) − IV have Δα t −0.74 and 0.56.

Cumulative paired daily differences. Top: pairs registered on the total difference. Bottom: β-adjusted (solid, $d_t - \hat\beta\,\mathrm{mkt}_t$) and total (dashed) for the pairs registered on Δα.

In [ ]:
alpha_pairs = [("ERC(S)", "GMV(S)"), ("ERC(S)", "MDP(S)"), ("ERC(S)", "IV")]
total_pairs = [("ERC(S)", "EW"), ("ERC(LW)", "ERC(S)")]

fig, axes = plt.subplots(2, 1, figsize=(9.5, 9), sharex=True)
ml.plotting.cumulative_paired(
    paired_series, total_pairs, split_ts, ax=axes[0],
    title="(a) registered on the total difference (H4, H5 LW)", ylabel="cumulative diff (%)",
)
ml.plotting.cumulative_paired(
    paired_series, alpha_pairs, split_ts,
    beta={f"{a} - {b}": diff_capm_table.loc[f"{a} - {b}", "beta_diff"] for a, b in alpha_pairs},
    mkt_excess=MKT_simple, ax=axes[1],
    title="(b) registered on Delta-alpha (H3, H5 IV): solid = beta-adjusted, dashed = total",
)
plt.tight_layout(); plt.show()

The endpoints match 17.2 years × the annualized means: in (a) ERC(S) − EW ends near −47.8% and ERC(LW) − ERC(S) near +3.2%; in (b) the totals against GMV, MDP and IV end near +17.9%, +6.8% and −17.0%, the β-adjusted lines near +1.6%, −4.5% and +5.6%. ERC − EW loses most in 2009 and 2020–21, when equities rallied. ERC − MDP's β-adjusted line turns negative in 2022.

## §7 What drove the results

### 7.1 Weights by asset group

Capital by asset group over time for ERC(S), MDP(S) and GMV(S). Below is the mean over the 208 rebalances, with EW and 60/40 for reference (%).

In [ ]:
fig = ml.plotting.group_stackplot(results, ["ERC(S)", "MDP(S)", "GMV(S)"], split_ts=split_ts)
plt.show()

group_means = {}
for name in ["ERC(S)", "MDP(S)", "GMV(S)", "EW", "60/40"]:
    g = ml.diagnostics.weights_by_group(results[name]["wlog"])
    assert len(g) == 208, (name, len(g))
    group_means[name] = g.mean()
group_means = pd.DataFrame(group_means)
print((100 * group_means).to_string(float_format=lambda v: f"{v:.1f}"))

On average ERC(S) holds 32.0% in fx (UUP) and 10.1% in equity; MDP(S) holds 44.8% and 13.6%, GMV(S) 37.0% and 7.0%. ERC(S) is the only one of the three with sizeable credit and inflation-linked weights (19.0% and 9.7%; MDP(S) 1.8% and 0.8%).

### 7.2 The UUP corner (H1)

UUP target weight under ERC(S) against GMV(S) and MDP(S) at the 208 rebalances (40 in test), with $\beta_{UUP,p}$ at 2022-12-31 as context.

In [ ]:
w_uup_erc_ts = results["ERC(S)"]["wlog"]["UUP"]
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]

lt_gmv = w_uup_erc_ts < w_uup_gmv_ts
lt_mdp = w_uup_erc_ts < w_uup_mdp_ts
h1_full = lt_gmv & lt_mdp
n_full = len(h1_full)
test_mask = h1_full.index > split_ts
h1_test = h1_full.loc[test_mask]
n_test = int(test_mask.sum())

asof_2022 = pd.Timestamp("2022-12-31")
uup_2022 = {
    "ERC(S)": float(w_uup_erc_ts.loc[asof_2022]),
    "GMV(S)": float(w_uup_gmv_ts.loc[asof_2022]),
    "MDP(S)": float(w_uup_mdp_ts.loc[asof_2022]),
}

_, Sigma_2022c = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np_2022c = Sigma_2022c.to_numpy()
w_erc_2022_arr = results["ERC(S)"]["wlog"].loc[asof_2022].reindex(Sigma_2022c.columns).to_numpy()
var_2022c = float(w_erc_2022_arr @ Sigma_np_2022c @ w_erc_2022_arr)
beta_p_2022c = (Sigma_np_2022c @ w_erc_2022_arr) / var_2022c
iU2022c = list(Sigma_2022c.columns).index("UUP")

print(f"H1 (ERC<GMV and ERC<MDP): full {int(h1_full.sum())}/{n_full} "
      f"({100*h1_full.sum()/n_full:.1f}%), test {int(h1_test.sum())}/{n_test} "
      f"({100*h1_test.sum()/n_test:.1f}%)")
print(f"2022-12-31 w_UUP: ERC={uup_2022['ERC(S)']:.4f}  GMV={uup_2022['GMV(S)']:.4f}  MDP={uup_2022['MDP(S)']:.4f}")
print(f"2022-12-31 beta_UUP,p (ERC) = {beta_p_2022c[iU2022c]:.4f}")
print(f"holds at 2022-12-31: ERC<GMV={uup_2022['ERC(S)'] < uup_2022['GMV(S)']}  "
      f"ERC<MDP={uup_2022['ERC(S)'] < uup_2022['MDP(S)']}")

mean_max_erc = float(results["ERC(S)"]["wlog"].max(axis=1).mean())
print(f"ERC(S) largest weight, mean over {n_full} rebalances: {mean_max_erc:.6f}")

ERC(S) holds less UUP than both GMV(S) and MDP(S) at 197/208 rebalances (33/40 test) and at 2022-12-31: 48.36% against 58.12% and 59.22%. $\beta_{UUP,p}$ = 0.1591 is positive, so UUP diversifies the ERC portfolio rather than hedging it, and by §3 a low beta needs a large capital weight to carry 1/13 of the variance. UUP is ERC(S)'s largest weight throughout (the max-weight and UUP panels in §7.3 coincide), and that weight averages 32.0%.

### 7.3 Concentration and turnover (H2)

Effective N of ERC(S) against the larger of GMV(S)'s and MDP(S)'s at the 208 rebalances (40 in test).

In [ ]:
effN_erc_ts = 1.0 / (results["ERC(S)"]["wlog"] ** 2).sum(axis=1)
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)

comparator_max = pd.concat([effN_gmv_ts, effN_mdp_ts], axis=1).max(axis=1)
h2_full = effN_erc_ts >= comparator_max
n_full2 = len(h2_full)
test_mask2 = h2_full.index > split_ts
h2_test = h2_full.loc[test_mask2]
n_test2 = int(test_mask2.sum())

print(f"H2 (effN(ERC) >= max(GMV,MDP)): full {int(h2_full.sum())}/{n_full2} "
      f"({100*h2_full.sum()/n_full2:.1f}%), test {int(h2_test.sum())}/{n_test2} "
      f"({100*h2_test.sum()/n_test2:.1f}%)")
print(f"median effN: ERC(S)={float(effN_erc_ts.median()):.4f}  "
      f"GMV(S)={float(effN_gmv_ts.median()):.4f}  MDP(S)={float(effN_mdp_ts.median()):.4f}")
print(f"2022-12-31 effN: ERC={float(effN_erc_ts.loc[asof_2022]):.4f}  "
      f"GMV={float(effN_gmv_ts.loc[asof_2022]):.4f}  MDP={float(effN_mdp_ts.loc[asof_2022]):.4f}")

Effective N, largest weight, UUP weight and trailing 12-rebalance one-way turnover (%/yr, initial allocation excluded); EW dashed, 60/40 as a reference line.

In [ ]:
fig = ml.plotting.concentration_panel(
    results, ["ERC(S)", "MDP(S)", "GMV(S)", "60/40"], split_ts=split_ts,
    rows=("effN", "max_w", "highlight", "turnover"), figsize=(9.5, 14),
    suptitle="ERC(S) vs MDP(S) vs GMV(S), 60/40 reference: effN, max weight, UUP weight, turnover",
)
plt.show()

ERC(S)'s effective N is at least the larger of GMV(S)'s and MDP(S)'s at 208/208 rebalances (40/40 test); medians 6.86, 3.79 and 3.35, but only 3.88 at 2022-12-31 (GMV 2.51, MDP 2.47): equal variance shares are not equal capital. Its trailing turnover stays between about 20% and 40%/yr, below MDP(S)'s and GMV(S)'s throughout.

## Verdict table

The §2 rules applied to the statistics above: H1 in §7.2, H2 in §7.3, H3–H5 in §6.2. The rule column shows the registered wording; `[applied: …]` states how an ambiguous rule was read.

In [ ]:
reg = ml.inference.load_registration("nb06")

h1_holds_2022 = (uup_2022["ERC(S)"] < uup_2022["GMV(S)"]) and (uup_2022["ERC(S)"] < uup_2022["MDP(S)"])
d4 = decomp["ERC(S) - EW"]
t_full_ew = float(paired_table.loc[("ERC(S) - EW", "full"), "t"])
mean_test_ew = float(paired_table.loc[("ERC(S) - EW", "test (> split)"), "ann_mean_diff"])
t_full_lw = float(paired_table.loc[("ERC(LW) - ERC(S)", "full"), "t"])
t_test_lw = float(paired_table.loc[("ERC(LW) - ERC(S)", "test (> split)"), "t"])

stats = {
    "H1": dict(holds_2022=h1_holds_2022, frac_full=h1_full.sum() / n_full, frac_test=h1_test.sum() / n_test,
               display=f"full {int(h1_full.sum())}/{n_full}, test {int(h1_test.sum())}/{n_test}, "
                       f"2022 holds={h1_holds_2022}"),
    "H2": dict(frac_full=h2_full.sum() / n_full2, frac_test=h2_test.sum() / n_test2,
               display=f"full {int(h2_full.sum())}/{n_full2}, test {int(h2_test.sum())}/{n_test2}"),
}
for hid, pair_key in [("H3_GMV", "ERC(S) - GMV(S)"), ("H3_MDP", "ERC(S) - MDP(S)"), ("H5_IV", "ERC(S) - IV")]:
    t = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])
    stats[hid] = dict(t=t, display=f"t={t:.2f}")
stats["H4"] = dict(t_full=t_full_ew, mean_test=mean_test_ew, beta_part=d4["beta_part"], d_alpha=d4["d_alpha"],
                   display=f"t_full={t_full_ew:.2f}, mean_test={mean_test_ew:.4f}, "
                           f"|beta_part|={abs(d4['beta_part']):.4f} vs |d_alpha|={abs(d4['d_alpha']):.4f}")
stats["H5_LW"] = dict(t_full=t_full_lw, t_test=t_test_lw, display=f"t_full={t_full_lw:.2f}, t_test={t_test_lw:.2f}")

vt = ml.inference.evaluate(reg, stats)
print(vt.to_string(index=False))

## §8 TAKEAWAY — equal risk shrinks the UUP corner but does not remove it

- **H1 supported: the UUP corner shrinks but survives.** ERC(S) holds less UUP than GMV(S) and MDP(S) at 197/208 rebalances (33/40 test); at 2022-12-31 the weights are 48.36% against 58.12% and 59.22%.
- **Why UUP still gets that much.** Every variance share is 1/13, so $w_i = \sigma_p/(N\sigma_i\rho_{i,p})$. UUP's low volatility and its low correlation with the portfolio ($\rho_{i,p}$ = 0.0741) both raise its weight; correlation by more. Its beta to the portfolio is positive: it diversifies, it does not hedge.
- **H2 supported.** Effective N is at least the larger of GMV's and MDP's at 208/208 rebalances (40/40 test). The median is 6.86 against 3.79 and 3.35, but only 3.88 at 2022-12-31.
- **Returns.** H3 nulls hold on Δα (t 0.29 and −0.74); ERC(S)'s gap over GMV(S) is mostly β. H4 supported: ERC(S) − EW is −2.78%/yr (t −2.08), β-driven. H5 nulls hold, LW narrowly (t 1.75).
- **Benchmarks (descriptive).** Full-window Sharpe ratio 0.76 against EW's 0.74 and 60/40's 0.91; tests are in notebook 08.

Equal risk shares cap UUP's share of variance at 1/13, not its share of capital.